In [3]:
# GOAL :: Do familail relationship regression (FR-reg)

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
from BIGFAM import frreg, tools
import importlib

In [2]:
source = "GS"

# Step 1. Load data

In [3]:
# parameters
info_fn = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/relative_information/relatives.formatted.info"

In [4]:
# relative information format
df_pair = pd.read_csv(info_fn, sep='\t')
df_pair.head()

,DOR,rcode,relationship,volid,relid,volage,relage,volsex,relsex,Erx
0,1,SB,daughter-sister,18826,21244,50,36,F,F,0.750000
1,1,SB,different-sex-sibling,34422,23884,33,35,F,M,0.353553
2,1,PC,daughter-mother,79198,67531,66,44,F,F,0.500000
3,1,SB,daughter-sister,20399,67531,38,44,F,F,0.750000
4,1,SB,daughter-sister,67267,67531,43,44,F,F,0.750000


In [5]:
# for rcode in ["PC", "SB"]:
#     tmp = df_pair[df_pair["rcode"] == rcode]
#     n_pair = len(tmp)
#     n_indiv = len(set(tmp["volid"].unique()) | set(tmp["relid"].unique()))
#     print(f"""
#           [{rcode}]
#           n_pair = {n_pair}
#           n_indiv = {n_indiv}
#           """)

# Step 2. FR-reg

In [6]:
pheno_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/phenotype"
frreg_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/frreg"

In [7]:
pheno_fns = os.listdir(pheno_path)
len(pheno_fns)

40

## Step 2.1 DOR level

In [20]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    try:
        df_frreg, msgs = frreg.familial_relationship_regression_DOR(df_mrg)
        
        if len(msgs) > 0:
            warning_dicts[pheno] = msgs
            continue
        
        df_frreg.to_csv(
            f"{frreg_path}/DOR/{pheno}.DOR.frreg",
            sep='\t',
            index=False
        )
    except:
        continue

100%|██████████| 40/40 [00:19<00:00,  2.01it/s]


## Step 2.2 REL level

In [20]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    
    try:
        df_frreg, msgs = frreg.familial_relationship_regression_REL(df_mrg)
        
        df_frreg_positive = (df_frreg[df_frreg["slope"] > 0]
                             .reset_index(drop=True))
        
        if len(msgs) > 0:
            print(fn, msgs)
            warning_dicts[pheno] = msgs
        
        if len(df_frreg_positive) < 10:
            print(fn, "not sufficient data")
            continue
    
        df_frreg_positive.to_csv(
            f"{frreg_path}/REL/{pheno}.REL.frreg",
            sep='\t',
            index=False
        )
    except:
        continue

  2%|▎         | 1/40 [00:01<00:53,  1.36s/it]

QT_interval.phen ['Some FR-reg slope is NEGATIVE']


  5%|▌         | 2/40 [00:02<00:51,  1.36s/it]

PR_interval.phen ['Some FR-reg slope is NEGATIVE']


 12%|█▎        | 5/40 [00:06<00:44,  1.26s/it]

...

 92%|█████████▎| 37/40 [00:39<00:03,  1.29s/it]

avg_dia.phen ['Some FR-reg slope is NEGATIVE']


 98%|█████████▊| 39/40 [00:41<00:00,  1.05it/s]

P_duration.phen ['Some FR-reg slope is NEGATIVE']
waist.phen not sufficient data


100%|██████████| 40/40 [00:42<00:00,  1.06s/it]


# cf. po/sib

In [3]:
pheno_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/phenotype"
frreg_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{source}/po-sib/frreg"

In [4]:
pheno_fns = os.listdir(pheno_path)
len(pheno_fns)

40

## Step 2.1 DOR level

In [14]:
warning_dicts = {}

for fn in tqdm(pheno_fns):
    pheno = fn.split(".")[0]
    pheno_fn = f"{pheno_path}/{fn}"
    df_pheno = pd.read_csv(pheno_fn, sep="\t")
    df_pheno = frreg.remove_outliers(df_pheno, "pheno")
    
    # merge pheno with relatives
    df_mrg = frreg.merge_pheno_info(df_pheno, df_pair)
    
    for exclude in ["PC", "SB"]:
        df_target = df_mrg[df_mrg["rcode"] != exclude].copy()
        
        try:
            df_frreg, msgs = frreg.familial_relationship_regression_DOR(df_target)
        
            
            if len(msgs) > 0:
                warning_dicts[pheno] = msgs
                continue
            
            if exclude == "PC":
                df_frreg.to_csv(
                    f"{frreg_path}/{pheno}.SB.frreg",
                    sep='\t',
                    index=False
                )
            elif exclude == "SB":
                df_frreg.to_csv(
                    f"{frreg_path}/{pheno}.PC.frreg",
                    sep='\t',
                    index=False
                )
        except:
            pass

100%|██████████| 40/40 [00:30<00:00,  1.31it/s]
